In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)

texts_transfers = [
    "перевод денег по номеру телефона",
    "не могу перевести средства на карту",
    "комиссия за перевод между счетами",
    "отправить деньги родственникам через приложение",
    "завис перевод по сбп средства не дошли",
    "повторить регулярный платеж и перевод",
    "лимит на суточный перевод в мобильном банке"
] * 20  # 140 примеров

texts_cards = [
    "заказ новой дебетовой карты онлайн",
    "перевыпуск пластика в отделении банка",
    "поменять пин код для дебетовой карты",
    "доставка карты курьером на дом"
] * 10  # 40 примеров

texts_fraud = [
    "срочно заблокируйте счет списали все мошенники",
    "украли деньги с карты подозрительные операции",
    "несанкционированный вход в личный кабинет взлом"
] * 7   # 21 пример

data = pd.DataFrame({
    "text": texts_transfers + texts_cards + texts_fraud,
    "label": [0] * len(texts_transfers) + [1] * len(texts_cards) + [2] * len(texts_fraud)
})

target_names = ["Transfers", "Cards", "Fraud"]

In [15]:
# =====================================================================
# ЗАДАНИЕ 1: Стратифицированный сплит и Dummy Baseline
# =====================================================================
# TODO 1:
# 1. Разбей data["text"] и data["label"] на train (80%) и test (20%).
#    Обязательно используй аргумент stratify=data["label"], random_state=42.
X_train, X_test, y_train, y_test = train_test_split(
    data['text'], 
    data['label'], 
    stratify=data['label'], 
    test_size=0.2, 
    random_state=42
    )

# 2. Создай DummyClassifier(strategy="most_frequent") и обучи его на X_train, y_train.
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train, y_train)

# 3. Сделай предсказания на X_test, посчитай Accuracy и Macro F1.
#    Убедись, почему высокая Accuracy здесь обманчива.
dummy_pred = dummy_model.predict(X_test)

print(accuracy_score(y_test, dummy_pred), '\n')
print(classification_report(y_test, dummy_pred))


0.7073170731707317 

              precision    recall  f1-score   support

           0       0.71      1.00      0.83        29
           1       0.00      0.00      0.00         8
           2       0.00      0.00      0.00         4

    accuracy                           0.71        41
   macro avg       0.24      0.33      0.28        41
weighted avg       0.50      0.71      0.59        41



c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [16]:

# =====================================================================
# ЗАДАНИЕ 2: Сравнение пайплайнов (MultinomialNB vs LogisticRegression)
# =====================================================================
# TODO 2:
# 1. Собери Pipeline для Наивного Байеса:
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ('clf', MultinomialNB())
])

# 2. Собери второй Pipeline для Логистической регрессии:
log_reg_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))
])

# 3. Обучи оба пайплайна на X_train, y_train.
nb_pipeline.fit(X_train, y_train)
log_reg_pipeline.fit(X_train, y_train)

# 4. Выведи classification_report для обоих.
#    Сравни Macro F1 моделей — какая лучше справляется с классом Fraud?
nb_pred = nb_pipeline.predict(X_test)
lr_pred = log_reg_pipeline.predict(X_test)

print(classification_report(y_test, nb_pred), '\n')
print(classification_report(y_test, lr_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        29
           1       1.00      1.00      1.00         8
           2       1.00      1.00      1.00         4

    accuracy                           1.00        41
   macro avg       1.00      1.00      1.00        41
weighted avg       1.00      1.00      1.00        41
 

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        29
           1       1.00      1.00      1.00         8
           2       1.00      1.00      1.00         4

    accuracy                           1.00        41
   macro avg       1.00      1.00      1.00        41
weighted avg       1.00      1.00      1.00        41



In [17]:
# =====================================================================
# ЗАДАНИЕ 3: Матрица ошибок и анализ важности признаков
# =====================================================================
# TODO 3:
# 1. Посчитай нормализованную по строкам матрицу ошибок для LogisticRegression:
cm = confusion_matrix(y_test, lr_pred, normalize='true')
print(np.round(cm, 2))

# 2. Интерпретация весов:
#    - Достань обученный классификатор из пайплайна
#    - Достань словарь признаков
#    - Вектор весов для класса Fraud (индекс 2)
#    - Найди индексы топ-5 слов с наибольшими положительными весами через 
#    - Выведи топ-5 маркеров мошенничества.
clf = log_reg_pipeline.named_steps['clf']

features_names = log_reg_pipeline.named_steps['tfidf'].get_feature_names_out()

weight2 = clf.coef_[2]

indices = np.argsort(weight2)[-5:][::-1]

print('Топ 5 маркеров мошеничества')
for i in indices:
    print(f"Признак - {features_names[i]:<25} - Вес {weight2[i]:.4}")

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
Топ 5 маркеров мошеничества
Признак - украли деньги             - Вес 0.8001
Признак - украли                    - Вес 0.8001
Признак - подозрительные операции   - Вес 0.8001
Признак - подозрительные            - Вес 0.8001
Признак - операции                  - Вес 0.8001
